# 24 · minitorch ↔ PyTorch：对照总表与毕业指南

> **Part 9 · 收尾。**

恭喜你走到这里！你已经从零造出了一个能训练 MLP / CNN / RNN / Transformer 的微型深度学习框架。本节把 **minitorch 与 PyTorch 的 API 一一对照**，并告诉你 PyTorch 在我们的基础上多做了什么、接下来该学什么。

## 我们一路造了什么

```
数学直觉 → 标量 autograd(Value) → 张量 autograd(Tensor，框架心脏) →
nn(Module/Linear/损失) → 优化器/DataLoader/正则化/归一化 →
CNN(Conv2d/Pool) → RNN/LSTM(BPTT/门控) → Transformer(注意力/多头/编码层)
```

每一块都**亲手实现 + 数值梯度验证 + PyTorch 对照**。你对深度学习"为什么有效"已经有了实现层面的理解。

## API 对照总表

minitorch 的 API 是**刻意贴着 PyTorch** 设计的，所以你的知识可以无缝迁移：

| 概念 | minitorch | PyTorch |
|---|---|---|
| 张量 | `minitorch.Tensor(x)` | `torch.tensor(x)` |
| 反向传播 | `loss.backward()` | `loss.backward()` |
| 不追踪梯度 | `with minitorch.no_grad():` | `with torch.no_grad():` |
| 基类 | `nn.Module` | `nn.Module` |
| 全连接 | `nn.Linear(i, o)` | `nn.Linear(i, o)` |
| 容器 | `nn.Sequential(...)` | `nn.Sequential(...)` |
| 激活 | `nn.ReLU()` / `x.relu()` | `nn.ReLU()` / `F.relu(x)` |
| 损失 | `nn.CrossEntropyLoss()` | `nn.CrossEntropyLoss()` |
| 优化器 | `optim.Adam(params, lr)` | `torch.optim.Adam(params, lr)` |
| 数据 | `data.DataLoader(ds, bs, shuffle)` | `DataLoader(ds, bs, shuffle)` |
| 卷积 | `nn.Conv2d(i, o, k)` | `nn.Conv2d(i, o, k)` |
| 池化 | `nn.MaxPool2d(k)` | `nn.MaxPool2d(k)` |
| 归一化 | `nn.BatchNorm1d / LayerNorm` | `nn.BatchNorm1d / LayerNorm` |
| 循环 | `nn.LSTM(i, h)` | `nn.LSTM(i, h)` |
| 注意力 | `nn.MultiheadAttention(d, h)` | `nn.MultiheadAttention(d, h)` |

## 同一个模型，两种写法

下面用 minitorch 和 PyTorch 写**同一个 MLP** 并各训练几步——你会发现几乎是逐行对应的。

In [ ]:
import numpy as np
import minitorch
from minitorch import Tensor

# 一个简单的二分类合成数据
np.random.seed(0)
X = np.random.randn(200, 4); y = (X[:, 0] + X[:, 1] > 0).astype(int)

# ---------- minitorch ----------
minitorch.set_seed(0)
m1 = minitorch.nn.Sequential(minitorch.nn.Linear(4, 16), minitorch.nn.ReLU(), minitorch.nn.Linear(16, 2))
opt1 = minitorch.optim.Adam(m1.parameters(), lr=1e-2)
lf1 = minitorch.nn.CrossEntropyLoss()
for _ in range(100):
    opt1.zero_grad(); lf1(m1(Tensor(X)), y).backward(); opt1.step()
acc1 = (m1(Tensor(X)).data.argmax(1) == y).mean()

# ---------- PyTorch ----------
import torch
torch.manual_seed(0)
m2 = torch.nn.Sequential(torch.nn.Linear(4, 16), torch.nn.ReLU(), torch.nn.Linear(16, 2))
opt2 = torch.optim.Adam(m2.parameters(), lr=1e-2)
lf2 = torch.nn.CrossEntropyLoss()
Xt = torch.tensor(X, dtype=torch.float32); yt = torch.tensor(y)
for _ in range(100):
    opt2.zero_grad(); lf2(m2(Xt), yt).backward(); opt2.step()
acc2 = (m2(Xt).argmax(1).numpy() == y).mean()

print(f"minitorch 准确率: {acc1*100:.1f}%   PyTorch 准确率: {acc2*100:.1f}%")
print("训练循环结构完全一致：zero_grad -> forward -> loss -> backward -> step")

## PyTorch 在我们之上多做了什么

我们的 minitorch 抓住了**核心原理**，但要做真实的大规模工程，PyTorch 还提供了：

- **GPU 加速**：把张量和计算搬到 CUDA，比纯 NumPy/CPU 快几个数量级。
- **更全的算子与高效 C++/CUDA 实现**：卷积、注意力等都有高度优化的底层 kernel。
- **`torch.compile` / JIT**：把动态图编译成高效融合 kernel。
- **自动混合精度 (AMP)**：用 fp16/bf16 省显存、提速。
- **分布式训练**：多卡 / 多机数据并行、模型并行。
- **庞大生态**：`torchvision`、`torchaudio`、HuggingFace `transformers`、海量预训练模型。

但所有这些都建立在你已经理解的同一套基石之上：**张量 + 自动求导 + 模块化的层与优化器**。

## 接下来学什么

- **优化与训练**：学习率调度、梯度裁剪、混合精度、更大 batch 的技巧。
- **现代架构**：Transformer 变体（GPT/BERT/ViT）、残差网络、注意力的高效实现（FlashAttention）。
- **生成模型**：VAE、GAN、扩散模型（本项目未覆盖，但你已具备读懂它们的基础）。
- **工程实践**：用 PyTorch 复现一篇论文；在真实数据集上训练并调优。
- **回看源码**：挑一个你最感兴趣的 PyTorch 模块，对照你的 minitorch 实现读它的源码。

> 推荐资料：斯坦福 CS231n / CS224n、《动手学深度学习》(d2l.ai)、Karpathy 的 "Neural Networks: Zero to Hero"。

## 结语

你不再是"调包侠"——你**亲手造过轮子**，理解每一行 `loss.backward()` 背后到底发生了什么。这种理解会让你在阅读论文、调试模型、设计架构时都更有底气。

**祝你在深度学习的道路上越走越远！** 🚀

> 想再练手？打开 `99_capstone_project.ipynb`，挑一个开放式项目亲手做做看。